<a href="https://colab.research.google.com/github/lava789/insurellm-compliance-rag/blob/main/%20rag_insurellm_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai gradio

In [ ]:
import os
from google.colab import userdata

# Read the Gemini API key from Colab Secrets
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if GEMINI_API_KEY:
    print(f"✅ Gemini API Key exists and begins {GEMINI_API_KEY[:8]}")
else:
    print("❌ Gemini API Key not set — check your Colab Secrets")

# Make it available as an environment variable (used by google-genai automatically)
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

✅ Gemini API Key exists and begins AQ.Ab8RN


In [ ]:
import os, subprocess

# ---- Config ----
WEEK = "week5"   # change to "week1" for the Week 1 version
REPO = "https://github.com/ed-donner/llm_engineering.git"

# Clean up previous attempt
for path in ["llm_engineering", "knowledge-base"]:
    if os.path.exists(path):
        subprocess.run(["rm", "-rf", path], check=True)

# Sparse + shallow clone: only download the knowledge-base folder
subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", REPO], check=True)
subprocess.run(["git", "-C", "llm_engineering", "sparse-checkout", "set", f"{WEEK}/knowledge-base"], check=True)

# Move it up to the working directory so paths match the notebook
os.rename(f"llm_engineering/{WEEK}/knowledge-base", "knowledge-base")
subprocess.run(["rm", "-rf", "llm_engineering"], check=True)

# Verify
print("✅ knowledge-base ready")
print("📁 Top level :", os.listdir("knowledge-base"))
print("🏢 company   :", os.listdir("knowledge-base/company"))
print("📑 contracts :", len(os.listdir("knowledge-base/contracts")), "files")
print("👥 employees :", len(os.listdir("knowledge-base/employees")), "files")
print("📦 products  :", os.listdir("knowledge-base/products"))

✅ knowledge-base ready
📁 Top level : ['company', 'employees', 'contracts', 'products']
🏢 company   : ['culture.md', 'careers.md', 'overview.md', 'about.md']
📑 contracts : 32 files
👥 employees : 32 files
📦 products  : ['Claimllm.md', 'Rellm.md', 'Homellm.md', 'Markellm.md', 'Carllm.md', 'Healthllm.md', 'Bizllm.md', 'Lifellm.md']


In [ ]:
import glob
from pathlib import Path

knowledge = {}

filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

print(f"✅ Loaded {len(knowledge)} employees")
print("Keys:", list(knowledge.keys()))

✅ Loaded 30 employees
Keys: ['carter', 'rodriguez', "o'brien", 'greene', 'martinez', 'lancaster', 'walker', 'kim', 'chen', 'thomson', 'liu', 'anderson', 'williams', 'spencer', 'johnson', 'brooks', 'zhang', 'rivera', 'park', 'wilson', 'foster', 'sharma', 'trenton', 'thompson', 'tran', 'blake', 'patel', 'harper', 'bishop', 'adams']


In [ ]:
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

print(f"✅ knowledge now has {len(knowledge)} entries total")
print("Product keys:", [k for k in knowledge.keys() if k.endswith("llm")])

✅ knowledge now has 38 entries total
Product keys: ['claimllm', 'rellm', 'homellm', 'markellm', 'carllm', 'healthllm', 'bizllm', 'lifellm']


In [ ]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [ ]:
#token
def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]

Quick Test — run this to confirm it works:

In [ ]:
print("Q1:", get_relevant_context("Who is lancaster?"))
print()
print("Q2:", get_relevant_context("Who is Lancaster and what is carllm?"))

Q1: ["# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on mar

In [ ]:
get_relevant_context("Who is lancaster?")

["# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market 

additional_context() (Formatting the Retrieved Text)

In [ ]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    return result

In [ ]:
print(additional_context("Who is Alex Lancaster?"))

The following additional context might be relevant in answering the user's question:

# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: Business Analyst at Edge Analytics  
  Prior to joini

In [ ]:
from google import genai
from google.genai import types

client = genai.Client()
MODEL = "gemini-3.6-flash"

def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)

    # Convert Gradio history to Gemini Content objects
    contents = []
    for turn in history:
        role = "user" if turn["role"] == "user" else "model"
        contents.append(
            types.Content(
                role=role,
                parts=[types.Part(text=turn["content"])]
            )
        )
    contents.append(
        types.Content(
            role="user",
            parts=[types.Part(text=message)]
        )
    )

    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(system_instruction=system_message)
    )
    return response.text

In [ ]:
for m in client.models.list():
    print(m.name)

In [ ]:
import gradio as gr

view = gr.ChatInterface(chat).launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3b0884b19970f580b2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
